# Binance Spot hourly klines for the initial USDT universe

This notebook collects closed `1h` Binance Spot klines for the 30-pair initial universe approved in `docs/methodology_decisions.md`. The requested period is fixed from `2017-01-01 00:00:00 UTC` through `2026-09-01 23:59:59.999 UTC`. Each pair starts at its first available Binance candle; observations are not synthesized before listing or across missing intervals.

For each pair, the notebook creates a dedicated directory under `data/raw/binance/spot/1h/`. The canonical raw artifact is a `.csv.gz` file that preserves Binance decimal fields as text. A derived Excel review copy contains `Klines`, `Metadata`, `Validation`, and `Missing Intervals` sheets, and a JSON sidecar records checksums and validation results. A timestamped collection manifest summarizes the execution. Final artifacts are validated and never overwritten.

## Research limitation

The universe was selected ex post using activity observed through 2026-09-01 and current symbol status. These data may support exploration and later protocol development, but the fixed universe must not be presented as historically known at earlier portfolio decision timestamps.

In [1]:
import importlib
from pathlib import Path
import sys

import pandas as pd


def find_repository_root(start: Path | None = None) -> Path:
    """Find the repository root from a notebook or project directory."""
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "AGENTS.md").exists() and (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Could not locate the repository root.")


REPOSITORY_ROOT = find_repository_root()
SOURCE_DIRECTORY = REPOSITORY_ROOT / "src"
if str(SOURCE_DIRECTORY) not in sys.path:
    sys.path.insert(0, str(SOURCE_DIRECTORY))

import tcc_crypto.binance_hourly_collection as collection_module

collection_module = importlib.reload(collection_module)
load_collection_config = collection_module.load_collection_config
run_collection = collection_module.run_collection

CONFIG_PATH = (
    REPOSITORY_ROOT
    / "configs"
    / "binance_selected_usdt_hourly_collection.json"
)
config = load_collection_config(CONFIG_PATH)
RETRY_SYMBOLS = (
    "SOLUSDT",
    "XRPUSDT",
    "SHIBUSDT",
    "DOGEUSDT",
    "BNBUSDT",
    "ADAUSDT",
)
print(f"Repository root: {REPOSITORY_ROOT}")
print(f"Configuration: {CONFIG_PATH}")
print(f"Retry scope: {', '.join(RETRY_SYMBOLS)}")

Repository root: /Users/marcelapsilverio/projetos/tcc_crypto
Configuration: /Users/marcelapsilverio/projetos/tcc_crypto/configs/binance_selected_usdt_hourly_collection.json
Retry scope: SOLUSDT, XRPUSDT, SHIBUSDT, DOGEUSDT, BNBUSDT, ADAUSDT


In [2]:
configuration_summary = pd.DataFrame(
    [
        ("Collection ID", config.collection_id),
        ("Market", config.market),
        ("Interval", config.interval),
        ("Requested start (UTC)", config.start_time_utc.isoformat()),
        ("Requested end (UTC)", config.end_time_utc.isoformat()),
        ("Selected pairs", len(config.symbols)),
        ("Output directory", str(REPOSITORY_ROOT / config.output_directory)),
        ("Source endpoint", f"{config.base_url}{config.kline_path}"),
    ],
    columns=["Field", "Value"],
)
display(configuration_summary)
display(
    pd.DataFrame(
        {
            "Selection Rank": range(1, len(config.symbols) + 1),
            "Trading Pair": config.symbols,
        }
    )
)

,Field,Value
0,Collection ID,binance_selected_usdt_hourly_2017_2026-09-01
1,Market,Binance Spot
2,Interval,1h
3,Requested start (UTC),2017-01-01T00:00:00+00:00
4,Requested end (UTC),2026-09-01T23:59:59.999000+00:00
5,Selected pairs,30
6,Output directory,/Users/marcelapsilverio/projetos/tcc_crypto/da...
7,Source endpoint,https://data-api.binance.vision/api/v3/klines


,Selection Rank,Trading Pair
0,1,BTCUSDT
1,2,FLOKIUSDT
2,3,ETHUSDT
3,4,BONKUSDT
4,5,SOLUSDT
5,6,XRPUSDT
6,7,SHIBUSDT
7,8,DOGEUSDT
8,9,BNBUSDT
9,10,ADAUSDT


## Run the collection

The following cell retries only `SOLUSDT`, `XRPUSDT`, `SHIBUSDT`, `DOGEUSDT`, `BNBUSDT`, and `ADAUSDT`. Collection resumes after the last page persisted in each pair checkpoint, while the original selection ranks are preserved in the new manifest. Failed pairs are recorded without deleting successful outputs.

In [3]:
collection_results, manifest_path = run_collection(
    config,
    REPOSITORY_ROOT,
    symbols=RETRY_SYMBOLS,
    progress_callback=print,
)
display(collection_results)
print(f"Collection manifest: {manifest_path}")

Completed 1/6: SOLUSDT (COLLECTED)
Completed 2/6: XRPUSDT (COLLECTED)
Completed 3/6: SHIBUSDT (COLLECTED)
Completed 4/6: DOGEUSDT (COLLECTED)
Completed 5/6: BNBUSDT (COLLECTED)
Completed 6/6: ADAUSDT (COLLECTED)


,Selection Rank,Trading Pair,Status,Rows,Missing Hours,Shortened Candles,First Open (UTC),Last Close (UTC),Raw CSV,Raw CSV SHA-256,Workbook,Workbook SHA-256,Metadata,Error
0,5,SOLUSDT,COLLECTED,53086,20,5,2020-08-11T06:00:00+00:00,2026-09-01T23:59:59.999000+00:00,/Users/marcelapsilverio/projetos/tcc_crypto/da...,57b665ad7080ec45bd3c227a0ec2bd5dcdf3c8ffb78e9d...,/Users/marcelapsilverio/projetos/tcc_crypto/da...,b67dd4f3db6c51c1ede2385a719ab663d2ee8ad90dda57...,/Users/marcelapsilverio/projetos/tcc_crypto/da...,None
1,6,XRPUSDT,COLLECTED,72936,88,9,2018-05-04T08:00:00+00:00,2026-09-01T23:59:59.999000+00:00,/Users/marcelapsilverio/projetos/tcc_crypto/da...,f2a95318041ede35bdb1a60a7f0756fb5c43839ef6a936...,/Users/marcelapsilverio/projetos/tcc_crypto/da...,7b6d66e8e249bcdfe6842b2c2e8993481b467c154e0870...,/Users/marcelapsilverio/projetos/tcc_crypto/da...,None
2,7,SHIBUSDT,COLLECTED,46566,7,2,2021-05-10T11:00:00+00:00,2026-09-01T23:59:59.999000+00:00,/Users/marcelapsilverio/projetos/tcc_crypto/da...,f9c4c5951a500b8bbb8356ab05e04d23dfd66188e5bca3...,/Users/marcelapsilverio/projetos/tcc_crypto/da...,287f72f704885ec98df1038802c0f884890ce01bdf3a29...,/Users/marcelapsilverio/projetos/tcc_crypto/da...,None
3,8,DOGEUSDT,COLLECTED,62728,44,8,2019-07-05T12:00:00+00:00,2026-09-01T23:59:59.999000+00:00,/Users/marcelapsilverio/projetos/tcc_crypto/da...,4cc8785253081e0c90eaf79489b12de11cc35f174a8ded...,/Users/marcelapsilverio/projetos/tcc_crypto/da...,c0196aef167766b7b708e674ad2ad1cfecd98df3fa7414...,/Users/marcelapsilverio/projetos/tcc_crypto/da...,None
4,9,BNBUSDT,COLLECTED,77203,122,13,2017-11-06T03:00:00+00:00,2026-09-01T23:59:59.999000+00:00,/Users/marcelapsilverio/projetos/tcc_crypto/da...,e195bddc5bdeeb53564aff222fca2c7d41bca18d7e17d2...,/Users/marcelapsilverio/projetos/tcc_crypto/da...,01382dac7f39db3da6aeaf52421d30486890da6354b4cf...,/Users/marcelapsilverio/projetos/tcc_crypto/da...,None
5,10,ADAUSDT,COLLECTED,73348,88,10,2018-04-17T04:00:00+00:00,2026-09-01T23:59:59.999000+00:00,/Users/marcelapsilverio/projetos/tcc_crypto/da...,7677d8781ed7dd9508aa8ffa1842fe22ec3a62d21c8705...,/Users/marcelapsilverio/projetos/tcc_crypto/da...,0d99036454ff15a36b7760c5ba90422221805e0563a6c9...,/Users/marcelapsilverio/projetos/tcc_crypto/da...,None


Collection manifest: /Users/marcelapsilverio/projetos/tcc_crypto/data/raw/binance/spot/1h/collection_manifest_20260905T181544320481Z.xlsx


In [4]:
status_summary = (
    collection_results["Status"]
    .value_counts(dropna=False)
    .rename_axis("Status")
    .reset_index(name="Pairs")
)
display(status_summary)

failed = collection_results.loc[collection_results["Status"] == "FAILED"]
pairs_requiring_gap_review = collection_results.loc[
    collection_results["Missing Hours"].fillna(0) > 0
]
if not pairs_requiring_gap_review.empty:
    print("Pairs with missing hourly intervals requiring review:")
    display(
        pairs_requiring_gap_review[
            ["Trading Pair", "Missing Hours", "Workbook"]
        ]
    )

pairs_with_shortened_candles = collection_results.loc[
    collection_results["Shortened Candles"].fillna(0) > 0
]
if not pairs_with_shortened_candles.empty:
    print("Pairs with shortened historical candles requiring review:")
    display(
        pairs_with_shortened_candles[
            ["Trading Pair", "Shortened Candles", "Workbook"]
        ]
    )

if not failed.empty:
    display(failed[["Trading Pair", "Error"]])
    raise RuntimeError(
        f"Collection finished with {len(failed)} failed pair(s). "
        "Review the manifest and rerun the collection cell."
    )

print("All configured pairs were collected or already existed.")

,Status,Pairs
0,COLLECTED,6


Pairs with missing hourly intervals requiring review:


,Trading Pair,Missing Hours,Workbook
0,SOLUSDT,20,/Users/marcelapsilverio/projetos/tcc_crypto/da...
1,XRPUSDT,88,/Users/marcelapsilverio/projetos/tcc_crypto/da...
2,SHIBUSDT,7,/Users/marcelapsilverio/projetos/tcc_crypto/da...
3,DOGEUSDT,44,/Users/marcelapsilverio/projetos/tcc_crypto/da...
4,BNBUSDT,122,/Users/marcelapsilverio/projetos/tcc_crypto/da...
5,ADAUSDT,88,/Users/marcelapsilverio/projetos/tcc_crypto/da...


Pairs with shortened historical candles requiring review:


,Trading Pair,Shortened Candles,Workbook
0,SOLUSDT,5,/Users/marcelapsilverio/projetos/tcc_crypto/da...
1,XRPUSDT,9,/Users/marcelapsilverio/projetos/tcc_crypto/da...
2,SHIBUSDT,2,/Users/marcelapsilverio/projetos/tcc_crypto/da...
3,DOGEUSDT,8,/Users/marcelapsilverio/projetos/tcc_crypto/da...
4,BNBUSDT,13,/Users/marcelapsilverio/projetos/tcc_crypto/da...
5,ADAUSDT,10,/Users/marcelapsilverio/projetos/tcc_crypto/da...


All configured pairs were collected or already existed.


## Interpretation and next checks

A missing interval inside the observed span is recorded as `REVIEW`; it is not filled in the canonical raw CSV. A candle whose close remains inside its nominal hour but occurs before the final millisecond is also preserved and marked `REVIEW`, with details in `Nonstandard Durations`. A close before the open or beyond the nominal hour remains a fatal validation failure. Time before a pair's first observed candle is treated as pre-listing history rather than missing data. The Excel workbooks are derived inspection artifacts and must not replace the CSV files as pipeline inputs.